# Notebook 03 — Showcasing RLM Recursive Power

This notebook demonstrates what makes Recursive Language Models fundamentally
different from a plain LLM call: **the model decomposes a large task into
sub-tasks, delegates each to a child agent, and aggregates the results**.

The [original RLM paper](https://arxiv.org/abs/2512.24601) shows that this
recursive decomposition lets a model effectively handle inputs that far exceed
its context window — a capability no single-shot LLM can match.

### What you will see

| Experiment | Why sub-agents are needed |
|---|---|
| **A — Hierarchical Summarization** | Summarisation is an LLM task — you cannot summarise with regex. The agent **must** call `rlm_call` or `llm_call` to produce section summaries. |
| **B — Natural-Language Comprehension Q&A** | Facts are embedded in natural prose (numbers written as words, no labels). The agent must have sub-agents **read and understand** each section to answer questions. |

### Why the previous approach failed

The original notebook 03 used `[FACT_N]: value` markers that were trivially
findable with regex — the agent wrote `re.findall(...)` and never needed
sub-agents. This redesign makes regex shortcuts **impossible**:

- Facts are written in natural language ("four hundred seventy-three million", not "$473M")
- No labels, no markers, no patterns to grep for
- Questions are paraphrased differently from the source text
- Summarization inherently requires LLM understanding

---
## Setup

In [1]:
import os
import sys
import json
import random
import pathlib
import importlib

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

LLAMA_BASE_URL = os.environ.get('LLAMA_BASE_URL', 'http://host-gateway:1337/v1')
LLAMA_MODEL    = os.environ.get('LLAMA_MODEL',    'local-model')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'not-needed')
CODE_EXECUTION_TIMEOUT_SECONDS = os.environ.get('CODE_EXECUTION_TIMEOUT_SECONDS')
CODE_EXECUTION_TIMEOUT_SECONDS = None if CODE_EXECUTION_TIMEOUT_SECONDS in {None, '', 'none', 'None'} else int(CODE_EXECUTION_TIMEOUT_SECONDS)

import rlm_smolagent as rlm_mod
rlm_mod = importlib.reload(rlm_mod)
RLMAgent = rlm_mod.RLMAgent

import rlm_visualizer as rlm_vis_mod
rlm_vis_mod = importlib.reload(rlm_vis_mod)
save_html = rlm_vis_mod.save_html
save_json = rlm_vis_mod.save_json
load_json = rlm_vis_mod.load_json

project_root = pathlib.Path(rlm_mod.__file__).resolve().parent.parent
log_dir = project_root / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)

def make_agent(max_depth=3, max_steps=12, verbose=True, capture_prompt_traces=True, execution_timeout_seconds=CODE_EXECUTION_TIMEOUT_SECONDS):
    return RLMAgent(
        base_url=LLAMA_BASE_URL,
        model_name=LLAMA_MODEL,
        api_key=OPENAI_API_KEY,
        max_depth=max_depth,
        max_steps=max_steps,
        verbose=verbose,
        capture_prompt_traces=capture_prompt_traces,
        execution_timeout_seconds=execution_timeout_seconds,
    )

print('Setup complete.')
print(f'Project root: {project_root}')
print(f'Log directory: {log_dir} (exists={log_dir.exists()})')

Setup complete.
Project root: /workspace
Log directory: /workspace/logs (exists=True)


## Helper: pretty-print the call tree

In [2]:
def print_tree(node: dict, indent: int = 0) -> None:
    prefix = '  ' * indent
    depth  = node.get('depth', '?')
    dur    = node.get('duration_s', '?')
    task   = node.get('task_preview', node.get('task', '')[:120])
    resp   = node.get('response_preview', node.get('response', '')[:120])
    ctx    = node.get('context_size', 0)
    steps  = len(node.get('agent_steps', []))
    reqs   = len(node.get('llm_requests', []))
    kids   = len(node.get('children', []))
    print(f'{prefix}[depth={depth}] ({dur}s) ctx={ctx:,}B  steps={steps}  llm_reqs={reqs}  children={kids}')
    print(f'{prefix}  task    : {task}')
    print(f'{prefix}  response: {resp}')
    for child in node.get('children', []):
        print_tree(child, indent + 1)

---
## Step 1 — Build the long corporate report

We construct a multi-section corporate annual report with **six distinct sections**.
Each section covers a different business area (financials, HR, R&D, operations,
customer experience, legal). Five specific facts are embedded **as natural
language prose** — no labels, no markers, no patterns. Numbers are written as
words where possible.

This makes it impossible for the agent to use regex or string search to find
the answers. It must use LLM sub-calls to read and comprehend each section.

In [3]:
# ── Ground-truth facts (what we'll verify against) ──
GROUND_TRUTH = {
    'Q1': {'question': 'What was the total quarterly revenue?',
           'answer': '$473.8M',
           'keywords': ['473', 'four hundred', 'seventy-three', 'million'],
           'section': 'Financial Performance'},
    'Q2': {'question': 'Who was appointed as the new Chief Technology Officer?',
           'answer': 'Dr. Ryland Grace',
           'keywords': ['ryland', 'grace'],
           'section': 'Human Resources & Talent'},
    'Q3': {'question': 'How many new patent applications were filed by R&D this year?',
           'answer': '42',
           'keywords': ['42', 'forty-two'],
           'section': 'Research & Development'},
    'Q4': {'question': 'What server uptime percentage did the infrastructure team achieve?',
           'answer': '99.97%',
           'keywords': ['99.97', 'ninety-nine point nine seven'],
           'section': 'Operations & Infrastructure'},
    'Q5': {'question': 'When is the next flagship product launch scheduled?',
           'answer': 'September 15, 2026',
           'keywords': ['september', 'fifteenth', '2026', 'twenty twenty-six'],
           'section': 'Customer Experience'},
}

print('Ground truth questions:')
for qid, info in GROUND_TRUTH.items():
    print(f"  {qid}: {info['question']}")
    print(f"        Expected: {info['answer']} (in section: {info['section']})")

Ground truth questions:
  Q1: What was the total quarterly revenue?
        Expected: $473.8M (in section: Financial Performance)
  Q2: Who was appointed as the new Chief Technology Officer?
        Expected: Dr. Ryland Grace (in section: Human Resources & Talent)
  Q3: How many new patent applications were filed by R&D this year?
        Expected: 42 (in section: Research & Development)
  Q4: What server uptime percentage did the infrastructure team achieve?
        Expected: 99.97% (in section: Operations & Infrastructure)
  Q5: When is the next flagship product launch scheduled?
        Expected: September 15, 2026 (in section: Customer Experience)


In [4]:
# ── Build the document ──
# Each section has 15-20 paragraphs of topical filler with ONE key fact
# embedded in natural language prose.

SECTIONS = {}

# ---------- SECTION 1: Financial Performance ----------
SECTIONS['Financial Performance'] = [
    "The fiscal year concluded with a strong performance across all revenue streams. "
    "Management attributed the gains to disciplined execution of the multi-year "
    "growth strategy and favourable market conditions in key geographies.",

    "Gross margins expanded by approximately one hundred and twenty basis points "
    "compared with the prior-year period, driven by a shift toward higher-margin "
    "software-as-a-service offerings and reduced dependency on legacy hardware sales.",

    "Operating expenses were tightly managed despite inflationary headwinds. The "
    "finance team implemented a zero-based budgeting process that identified over "
    "fifteen million dollars in discretionary savings.",

    "Cash flow from operations remained robust, with free cash flow conversion "
    "exceeding ninety percent for the third consecutive quarter. The treasury team "
    "refinanced a portion of the long-term debt at more favourable rates.",

    "Capital expenditure was in line with the board-approved budget, focused "
    "primarily on expanding data-centre capacity and upgrading the global ERP "
    "platform to support the next phase of digital transformation.",

    "Total quarterly revenue reached four hundred and seventy-three point eight "
    "million dollars, reflecting a year-over-year increase of eleven percent. The "
    "enterprise segment contributed the largest share of the improvement, buoyed "
    "by a record number of large-deal closings during the quarter.",

    "Foreign-exchange impacts were largely neutral for the period. Currency "
    "hedging programmes offset the majority of adverse movements in the euro and "
    "yen, while the British pound provided a modest tailwind.",

    "Revenue guidance for the coming quarter incorporates a cautious outlook on "
    "macroeconomic conditions while factoring in the pipeline of committed deals. "
    "The CFO reiterated the company's commitment to delivering consistent earnings "
    "growth within the previously communicated range.",

    "Investor relations reported increased institutional interest following the "
    "earnings announcement, with several large asset managers adding to their "
    "positions. The stock repurchase programme continued as planned.",

    "Regional performance was mixed. The Americas delivered above-plan results, "
    "Asia-Pacific tracked slightly ahead, and EMEA faced continued softness in "
    "certain southern-European markets.",

    "The audit committee confirmed that all financial reporting controls operated "
    "effectively during the period. No material weaknesses were identified in the "
    "latest assessment of internal controls over financial reporting.",

    "Deferred revenue grew by nine percent, signalling sustained demand for "
    "multi-year subscription contracts. The bookings-to-billings ratio remained "
    "above one for the seventh straight quarter.",

    "Tax planning initiatives reduced the effective tax rate by approximately "
    "two percentage points relative to the statutory rate. Credits related to "
    "research and development activities were the primary contributor.",

    "Depreciation and amortisation charges were in line with expectations, "
    "reflecting the ongoing capitalisation of internally developed software and "
    "the recent acquisition of intellectual-property assets.",

    "Looking ahead, the finance team plans to explore additional debt-reduction "
    "opportunities and to accelerate the share-buyback programme if cash "
    "generation continues to exceed projections.",
]

# ---------- SECTION 2: Human Resources & Talent ----------
SECTIONS['Human Resources & Talent'] = [
    "The talent-acquisition function processed a record number of applications "
    "during the year while maintaining a consistently high quality bar. Time-to-fill "
    "for critical engineering roles decreased by twelve percent.",

    "Diversity and inclusion metrics showed continued progress. Representation of "
    "women in senior leadership roles increased to thirty-four percent, and the "
    "company earned recognition from several external benchmarking organisations.",

    "After an extensive global search, the board announced that Dr. Ryland Grace, "
    "a distinguished technologist with two decades of experience in artificial "
    "intelligence and distributed systems, would assume the role of Chief "
    "Technology Officer effective the first of the coming fiscal quarter.",

    "Employee engagement surveys indicated an overall satisfaction score in the "
    "top quartile of the industry benchmark. The areas of highest improvement were "
    "career-development opportunities and manager effectiveness.",

    "Voluntary attrition remained below the industry average, with the most "
    "significant retention gains observed in the product-engineering organisation. "
    "Exit-interview analysis pointed to competitive compensation and compelling "
    "project assignments as the primary retention drivers.",

    "Learning and development investment increased by eighteen percent. The "
    "newly launched internal academy offered over two hundred courses spanning "
    "technical skills, leadership, and business acumen.",

    "The internship programme expanded to twelve additional university partners, "
    "bringing the total to forty-seven globally. Conversion rates from intern to "
    "full-time employee remained above sixty percent.",

    "Compensation benchmarking confirmed that total-reward packages are positioned "
    "at or above the median for comparable technology companies. Equity-refresh "
    "grants were distributed to high-performing individual contributors and managers.",

    "Workplace-flexibility policies were formalised into a hybrid model that allows "
    "most roles to work remotely up to three days per week. Office-space utilisation "
    "data informed decisions to consolidate two smaller facilities.",

    "The wellness programme expanded to include mental-health resources, fitness "
    "subsidies, and an enhanced employee-assistance programme. Participation rates "
    "increased by twenty-two percent over the prior year.",

    "Succession planning for critical roles was reviewed by the board. Development "
    "plans are in place for all identified high-potential leaders, with regular "
    "check-ins scheduled quarterly.",

    "The HR technology stack migration progressed on schedule. The new cloud-based "
    "human-capital-management platform is expected to go live in the first quarter "
    "of the next fiscal year.",

    "Global mobility cases increased as the company expanded into three new "
    "international markets. Immigration support and relocation services were "
    "enhanced to ensure a smooth transition for relocating employees.",

    "Labour-relations activities remained constructive in all jurisdictions. "
    "Collective-bargaining agreements were renewed in two European subsidiaries "
    "without disruption.",

    "The HR team partnered with finance to model the workforce plan for the "
    "coming fiscal year. Headcount projections reflect targeted growth in "
    "engineering and go-to-market functions.",
]

# ---------- SECTION 3: Research & Development ----------
SECTIONS['Research & Development'] = [
    "The research division continued to advance the company's innovation pipeline, "
    "with several multi-year programmes reaching key inflection points during the "
    "year. R&D expenditure as a percentage of revenue increased modestly.",

    "Work on the next-generation platform architecture accelerated. The team "
    "completed a successful proof-of-concept that demonstrated a forty-percent "
    "reduction in inference latency for production workloads.",

    "The intellectual-property portfolio grew significantly. Over the course of "
    "the fiscal year, the research and development division submitted forty-two "
    "new patent applications, surpassing the previous annual record by twelve.",

    "Open-source contributions remained a strategic priority. The engineering "
    "team released four new libraries and maintained active involvement in several "
    "community-driven projects across the machine-learning ecosystem.",

    "Academic partnerships expanded to include three additional tier-one "
    "universities. Joint research programmes in quantum computing and advanced "
    "materials yielded two co-authored papers accepted at leading conferences.",

    "The advanced-materials team completed characterisation studies on a novel "
    "composite that could reduce manufacturing costs by up to twenty percent for "
    "certain product lines. Pilot production is scheduled for the next fiscal year.",

    "Internal hackathon events continued to surface innovative ideas. Three "
    "winning projects from the most recent hackathon were fast-tracked into the "
    "formal product-development pipeline.",

    "The biotechnology division launched a new exploratory programme focused on "
    "bio-inspired computing architectures. Early results suggest promising "
    "efficiency gains for specific workload categories.",

    "Testing infrastructure was upgraded to support continuous integration at "
    "scale. The new environment reduced end-to-end build-and-test cycle times "
    "from eight hours to under ninety minutes.",

    "The chief scientist presented the annual technology-roadmap update to the "
    "board of directors. Key themes for the coming three years include edge "
    "computing, federated learning, and privacy-preserving analytics.",

    "Licensing revenue from the patent portfolio contributed a growing share "
    "of total R&D returns. The IP-commercialisation team negotiated several "
    "new cross-licensing agreements with industry peers.",

    "Recruitment for the newly established AI safety research group is underway. "
    "The team will focus on alignment, interpretability, and adversarial-robustness "
    "testing for production-scale language models.",

    "Lab-equipment refresh was completed across all major research sites. "
    "New instruments include high-throughput sequencers, advanced electron "
    "microscopes, and a dedicated cluster for large-scale simulation workloads.",

    "R&D governance processes were updated to improve stage-gate discipline and "
    "ensure earlier identification of underperforming projects. The revised "
    "framework is expected to improve overall portfolio return on investment.",

    "Looking ahead, the research leadership team plans to increase investment "
    "in applied-AI research by thirty percent, reflecting the strategic importance "
    "of intelligent automation across the product suite.",
]

# ---------- SECTION 4: Operations & Infrastructure ----------
SECTIONS['Operations & Infrastructure'] = [
    "The infrastructure organisation delivered another year of reliable service "
    "across all production environments. Capacity planning models were refined to "
    "better anticipate seasonal demand spikes.",

    "Cloud migration reached a significant milestone, with seventy-eight percent "
    "of all workloads now running in the primary public-cloud provider. The "
    "remaining legacy data-centre footprint is scheduled for decommission by "
    "mid-next-year.",

    "The network-operations team upgraded backbone links between the three main "
    "availability zones, doubling throughput capacity and reducing inter-zone "
    "latency by eighteen percent.",

    "Over the full year the infrastructure team achieved a production server "
    "uptime of ninety-nine point nine seven percent, the highest figure in the "
    "company's history and well above the ninety-nine point nine percent target "
    "set by the service-level agreement.",

    "Incident-response processes were refined following a thorough review of "
    "all severity-one events. Mean time to detection fell by twenty-three percent "
    "and mean time to recovery improved by fifteen percent.",

    "Security posture assessments were conducted quarterly. Penetration testing "
    "by an independent firm identified only low-severity findings, all of which "
    "were remediated within the agreed-upon timeframe.",

    "Disaster-recovery drills were completed for all critical services. The "
    "exercise validated that recovery-time objectives can be met even under "
    "worst-case scenario assumptions.",

    "The platform-engineering team adopted an internal developer-platform model "
    "that standardised deployment pipelines and reduced configuration drift across "
    "micro-service environments.",

    "Observability tooling was consolidated onto a single platform. The new "
    "stack provides unified logging, metrics, and distributed tracing for all "
    "production services.",

    "Cost optimisation initiatives in the cloud environment yielded annualised "
    "savings of approximately four point five million dollars through right-sizing, "
    "reserved instances, and improved auto-scaling policies.",

    "Edge-computing pilots were launched in two geographic regions to evaluate "
    "latency reduction for real-time data-processing use cases. Early metrics "
    "indicate a meaningful improvement in end-user response times.",

    "The DevSecOps programme matured significantly. Static-analysis and container-"
    "scanning tools are now integrated into every CI/CD pipeline, catching "
    "vulnerabilities before code reaches production.",

    "Database administration practices were modernised with the adoption of "
    "automated schema-migration tooling and blue-green deployment patterns for "
    "zero-downtime releases.",

    "On-call rotations were restructured to improve engineer well-being. The "
    "new model distributes after-hours coverage more evenly and provides "
    "compensatory time off following busy rotations.",

    "The infrastructure roadmap for the coming year prioritises further cloud "
    "migration, zero-trust network architecture, and the rollout of confidential-"
    "computing capabilities.",
]

# ---------- SECTION 5: Customer Experience ----------
SECTIONS['Customer Experience'] = [
    "Customer-engagement metrics continued to trend positively. Monthly active "
    "users grew by fourteen percent year over year, with the strongest gains "
    "in the mobile channel.",

    "Net Promoter Score reached seventy-two, an all-time high for the company. "
    "Qualitative feedback highlighted the improved onboarding flow and responsive "
    "support experience as key satisfaction drivers.",

    "The support organisation resolved ninety-one percent of tickets within the "
    "first contact, up from eighty-four percent in the prior year. Average handle "
    "time decreased thanks to enhanced knowledge-base tooling.",

    "Product-led growth initiatives showed encouraging results. Free-to-paid "
    "conversion rates improved by three percentage points following the "
    "introduction of a redesigned trial experience.",

    "The product team confirmed that the next flagship product launch is "
    "scheduled for September fifteenth, twenty twenty-six. The new release "
    "will incorporate real-time collaboration features and an AI-powered "
    "assistant built on the company's proprietary model stack.",

    "User-research studies conducted across five markets revealed strong demand "
    "for deeper analytics capabilities and additional third-party integrations. "
    "These insights are informing the roadmap for the next two release cycles.",

    "The localisation programme expanded support to seven additional languages, "
    "bringing the total to twenty-three. Translation quality scores maintained "
    "their historical high.",

    "Customer-advisory-board sessions provided valuable strategic input. Feedback "
    "from large enterprise clients shaped several changes to the security-"
    "configuration interface and the audit-log export functionality.",

    "The marketplace ecosystem grew by twenty-eight certified partner integrations "
    "during the year. Partner-sourced deals now represent a meaningful portion of "
    "total new business.",

    "Customer-success teams adopted a proactive health-scoring model that "
    "identifies at-risk accounts earlier. Renewal rates for the enterprise tier "
    "improved by two percentage points quarter over quarter.",

    "Accessibility improvements were rolled out to meet WCAG 2.2 AA compliance. "
    "An internal audit verified that all customer-facing surfaces now meet the "
    "updated standard.",

    "The community forum saw a forty-percent increase in monthly active "
    "contributors. A dedicated community-manager role was created to sustain "
    "this growth and ensure constructive engagement.",

    "Churn analysis identified integration complexity as the top driver of "
    "cancellations among mid-market customers. A simplified integration wizard "
    "is being developed in response.",

    "Beta-programme participation doubled compared with the prior year. Early "
    "adopter feedback has been instrumental in refining the upcoming flagship "
    "release before its general-availability date.",

    "The customer-experience team plans to invest in predictive-support "
    "capabilities that surface solutions proactively, further reducing the need "
    "for reactive ticket creation.",
]

# ---------- SECTION 6: Legal & Compliance ----------
SECTIONS['Legal & Compliance'] = [
    "The legal department completed all planned regulatory-compliance audits "
    "without material findings. External counsel confirmed that the company's "
    "compliance posture meets or exceeds industry standards.",

    "Data-privacy programmes were strengthened in response to evolving legislation "
    "in the European Union, Brazil, and several US states. Privacy-impact "
    "assessments are now required for all new product features.",

    "Contract-management processes were streamlined through the adoption of a "
    "clause-library approach. Negotiation cycle times for standard agreements "
    "decreased by twenty-five percent.",

    "Litigation exposure remained low and manageable. The general counsel reported "
    "no pending matters expected to have a material adverse effect on the company's "
    "financial position.",

    "The ethics-and-compliance hotline received a modest number of reports, all of "
    "which were investigated and resolved in accordance with company policy. No "
    "systemic issues were identified.",

    "Anti-corruption training was completed by one hundred percent of employees "
    "in scope. The programme was updated to reflect recent enforcement trends and "
    "regulatory guidance.",

    "Intellectual-property protection efforts continued with the registration of "
    "trademarks in three new jurisdictions and the renewal of key domain-name "
    "portfolios.",

    "The board's risk committee reviewed the enterprise-risk register quarterly. "
    "Emerging risks related to supply-chain concentration and geopolitical "
    "instability were added to the watch list.",

    "Export-control compliance was verified through an independent assessment. "
    "The company's automated screening system processed a high volume of "
    "transactions without flagging any true-positive restricted-party matches.",

    "The legal-operations team rolled out e-billing and matter-management "
    "software, improving visibility into outside-counsel spending and enabling "
    "more rigorous invoice review.",

    "Insurance coverage was renewed at competitive terms. The risk-transfer "
    "strategy was updated to reflect the company's growth in new product "
    "categories and geographic markets.",

    "A comprehensive records-retention review was completed. Outdated retention "
    "schedules were updated and employee training on document-handling procedures "
    "was refreshed.",

    "The legal team contributed to industry working groups on responsible AI "
    "governance. Position papers authored by the team were cited by two "
    "regulatory bodies.",

    "Looking ahead, the legal department plans to invest in AI-assisted contract "
    "review tools and to expand the compliance-analytics function to cover "
    "additional regulatory domains.",
]

# ── Assemble the full report ──
report_parts = []
for section_name, paragraphs in SECTIONS.items():
    report_parts.append(f'\n{"=" * 60}\nSECTION: {section_name}\n{"=" * 60}')
    for p in paragraphs:
        report_parts.append(p)

REPORT = '\n\n'.join(report_parts)

print(f'Document length  : {len(REPORT):,} characters')
print(f'Approx. words    : {len(REPORT.split()):,}')
print(f'Sections         : {len(SECTIONS)}')
print(f'Total paragraphs : {sum(len(p) for p in SECTIONS.values())}')
print()
# Show first 400 chars from each section as preview
for name, paras in SECTIONS.items():
    combined = ' '.join(paras)
    print(f'  [{name}] {combined[:150]}...')

Document length  : 18,982 characters
Approx. words    : 2,353
Sections         : 6
Total paragraphs : 89

  [Financial Performance] The fiscal year concluded with a strong performance across all revenue streams. Management attributed the gains to disciplined execution of the multi-...
  [Human Resources & Talent] The talent-acquisition function processed a record number of applications during the year while maintaining a consistently high quality bar. Time-to-f...
  [Research & Development] The research division continued to advance the company's innovation pipeline, with several multi-year programmes reaching key inflection points during...
  [Operations & Infrastructure] The infrastructure organisation delivered another year of reliable service across all production environments. Capacity planning models were refined t...
  [Customer Experience] Customer-engagement metrics continued to trend positively. Monthly active users grew by fourteen percent year over year, with the strongest 

In [5]:
# Synchronise answer-bearing report paragraphs from GROUND_TRUTH so the
# evaluator and the filler document stay aligned after edits.
from datetime import datetime

ONES = {
    0: 'zero', 1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five',
    6: 'six', 7: 'seven', 8: 'eight', 9: 'nine', 10: 'ten', 11: 'eleven',
    12: 'twelve', 13: 'thirteen', 14: 'fourteen', 15: 'fifteen',
    16: 'sixteen', 17: 'seventeen', 18: 'eighteen', 19: 'nineteen',
}
TENS = {
    20: 'twenty', 30: 'thirty', 40: 'forty', 50: 'fifty',
    60: 'sixty', 70: 'seventy', 80: 'eighty', 90: 'ninety',
}
ORDINALS = {
    1: 'first', 2: 'second', 3: 'third', 4: 'fourth', 5: 'fifth',
    6: 'sixth', 7: 'seventh', 8: 'eighth', 9: 'ninth', 10: 'tenth',
    11: 'eleventh', 12: 'twelfth', 13: 'thirteenth', 14: 'fourteenth',
    15: 'fifteenth', 16: 'sixteenth', 17: 'seventeenth', 18: 'eighteenth',
    19: 'nineteenth', 20: 'twentieth', 21: 'twenty-first', 22: 'twenty-second',
    23: 'twenty-third', 24: 'twenty-fourth', 25: 'twenty-fifth',
    26: 'twenty-sixth', 27: 'twenty-seventh', 28: 'twenty-eighth',
    29: 'twenty-ninth', 30: 'thirtieth', 31: 'thirty-first',
}

def int_to_words(value: int) -> str:
    if value < 20:
        return ONES[value]
    if value < 100:
        tens = (value // 10) * 10
        remainder = value % 10
        return TENS[tens] if remainder == 0 else f"{TENS[tens]}-{ONES[remainder]}"
    if value < 1000:
        hundreds = value // 100
        remainder = value % 100
        prefix = f"{ONES[hundreds]} hundred"
        return prefix if remainder == 0 else f"{prefix} and {int_to_words(remainder)}"
    if value < 10000:
        thousands = value // 1000
        remainder = value % 1000
        prefix = f"{ONES[thousands]} thousand"
        if remainder == 0:
            return prefix
        joiner = ' and ' if remainder < 100 else ' '
        return f"{prefix}{joiner}{int_to_words(remainder)}"
    raise ValueError(f'Unsupported value for int_to_words: {value}')

def decimal_digits_to_words(text: str) -> str:
    return ' '.join(ONES[int(char)] for char in text if char.isdigit())

def money_to_report_words(answer: str) -> str:
    cleaned = answer.strip().replace('$', '').replace('M', '').replace('m', '')
    whole_text, dot, fraction_text = cleaned.partition('.')
    phrase = int_to_words(int(whole_text))
    if dot and fraction_text:
        phrase += f" point {decimal_digits_to_words(fraction_text)}"
    return f"{phrase} million dollars"

def percent_to_report_words(answer: str) -> str:
    cleaned = answer.strip().replace('%', '')
    whole_text, dot, fraction_text = cleaned.partition('.')
    phrase = int_to_words(int(whole_text))
    if dot and fraction_text:
        phrase += f" point {decimal_digits_to_words(fraction_text)}"
    return f"{phrase} percent"

def year_to_report_words(year: int) -> str:
    if 2000 <= year < 2100:
        suffix = year % 100
        return 'two thousand' if suffix == 0 else f"twenty {int_to_words(suffix)}"
    return int_to_words(year)

def date_to_report_words(answer: str) -> str:
    parsed = datetime.strptime(answer.strip(), '%B %d, %Y')
    return f"{parsed.strftime('%B')} {ORDINALS[parsed.day]}, {year_to_report_words(parsed.year)}"

derived_fact_text = {
    'revenue': money_to_report_words(GROUND_TRUTH['Q1']['answer']),
    'cto_name': GROUND_TRUTH['Q2']['answer'],
    'patent_count': int_to_words(int(GROUND_TRUTH['Q3']['answer'])),
    'uptime': percent_to_report_words(GROUND_TRUTH['Q4']['answer']),
    'launch_date': date_to_report_words(GROUND_TRUTH['Q5']['answer']),
}

SECTIONS['Financial Performance'][5] = (
    f"Total quarterly revenue reached {derived_fact_text['revenue']}, reflecting a "
    "year-over-year increase of eleven percent. The enterprise segment contributed "
    "the largest share of the improvement, buoyed by a record number of large-deal "
    "closings during the quarter."
)
SECTIONS['Human Resources & Talent'][2] = (
    f"After an extensive global search, the board announced that {derived_fact_text['cto_name']}, "
    "a distinguished technologist with two decades of experience in artificial "
    "intelligence and distributed systems, would assume the role of Chief "
    "Technology Officer effective the first of the coming fiscal quarter."
)
SECTIONS['Research & Development'][2] = (
    f"The intellectual-property portfolio grew significantly. Over the course of the "
    f"fiscal year, the research and development division submitted {derived_fact_text['patent_count']} "
    "new patent applications, surpassing the previous annual record by twelve."
)
SECTIONS['Operations & Infrastructure'][3] = (
    f"Over the full year the infrastructure team achieved a production server uptime "
    f"of {derived_fact_text['uptime']}, the highest figure in the company's history "
    "and well above the ninety-nine point nine percent target set by the service-level "
    "agreement."
)
SECTIONS['Customer Experience'][4] = (
    f"The product team confirmed that the next flagship product launch is scheduled "
    f"for {derived_fact_text['launch_date']}. The new release will incorporate real-time "
    "collaboration features and an AI-powered assistant built on the company's "
    "proprietary model stack."
)

report_parts = []
for section_name, paragraphs in SECTIONS.items():
    report_parts.append(f'\n{"=" * 60}\nSECTION: {section_name}\n{"=" * 60}')
    for paragraph in paragraphs:
        report_parts.append(paragraph)

REPORT = '\n\n'.join(report_parts)

print('Synced report facts from GROUND_TRUTH:')
print(f"  Revenue: {derived_fact_text['revenue']}")
print(f"  CTO    : {derived_fact_text['cto_name']}")
print(f"  Patents: {derived_fact_text['patent_count']}")
print(f"  Uptime : {derived_fact_text['uptime']}")
print(f"  Launch : {derived_fact_text['launch_date']}")

Synced report facts from GROUND_TRUTH:
  Revenue: four hundred and seventy-three point eight million dollars
  CTO    : Dr. Ryland Grace
  Patents: forty-two
  Uptime : ninety-nine point nine seven percent
  Launch : September fifteenth, twenty twenty-six


---
## Experiment A — Hierarchical Summarization

This is the **classic RLM demonstration** from the paper. Summarisation
**inherently requires LLM understanding** — you cannot summarise with regex
or string search. The agent must:

1. Split the document into sections (Python logic)
2. Call `rlm_call` or `llm_call` on each section to produce a summary
3. Aggregate the summaries into a coherent executive overview

This creates a clear, multi-level call tree that demonstrates recursive
decomposition in action.

In [6]:
agent_summ = make_agent(max_depth=2, max_steps=12, verbose=True)

summ_result = agent_summ.completion(
    task=(
        "`rlm_context` is a multi-section corporate annual report. "
        "Each section begins with a header block: a line of '=' characters, then 'SECTION: Name', then another line of '=' characters.\n\n"
        "Your task: Produce a **concise** executive summary — ONE short paragraph "
        "(3-5 sentences) per section. Each paragraph must include the section name "
        "and 1-2 of the most important numbers, dates, or metrics from that section. "
        "Do NOT include every detail — only the single most impactful finding per section.\n\n"
        "The final output must be **under 1500 characters total**.\n\n"
        "Strategy (you MUST follow this):\n"
        "1. Use Python to split `rlm_context` into sections. Look for lines starting with 'SECTION:' to identify section boundaries.\n"
        "2. For each section, call llm_call(task, section_text) where task asks the sub-agent "
        "   to return a single short paragraph (3-5 sentences) with only the most important finding. "
        "   This is REQUIRED — you cannot summarise without LLM comprehension.\n"
        "3. Combine all section summaries into a final executive summary.\n\n"
        "IMPORTANT:\n"
        "- You MUST use llm_call() or rlm_call() to process each section.\n"
        "- Do NOT try to summarise by reading the text yourself — delegate to sub-agents.\n"
        "- Pass section content as context_slice, NOT embedded in the task string.\n"
        "- Keep each sub-agent's output SHORT — 3-5 sentences maximum per section.\n"
    ),
    context=REPORT,
)

print('\n' + '=' * 60)
print('EXECUTIVE SUMMARY')
print('=' * 60)
print(summ_result.response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are an RLM (Recursive Language Model) agent at recursion depth 0/2.                                         │
│                                                                                                                 │
│ You run inside a Python REPL.  The input context is available as the                                            │
│ Python variable `rlm_context` — treat it as a Python object.  Slice it,                                         │
│ search it, split it, transform it.  Do NOT embed its raw content as a                                           │
│ string literal inside any sub-call argument.                                                                    │
│                                                                                                                 │
│ Two tools are available for making LLM sub-calls:                                                               │
│                                                                                                                 │
│ `llm_call(sub_task, context_slice)`:                                                                            │
│     Direct, non-recursive LLM call.  Fast and lightweight.                                                      │
│     Use for leaf-level queries on chunks that are small enough to                                               │
│     answer in a single LLM call without further decomposition.                                                  │
│                                                                                                                 │
│ `rlm_call(sub_task, context_slice)`:                                                                            │
│     Recursive RLM sub-call.  The child agent gets its own Python REPL                                           │
│     and can decompose the slice further.  Use for complex sub-tasks                                             │
│     that may themselves need recursive processing.                                                              │
│                                                                                                                 │
│ You decide HOW to orchestrate — use any Python logic to split, filter,                                          │
│ or transform `rlm_context` before passing slices to sub-calls.                                                  │
│                                                                                                                 │
│ Example — summarise paragraph-by-paragraph with direct LLM calls:                                               │
│     paragraphs = [p for p in rlm_context.split("\n\n") if p.strip()\]                                           │
│     summaries = [llm_call(f"Summarise paragraph {i+1}", p)                                                      │
│                  for i, p in enumerate(paragraphs)\]                                                            │
│     final_answer("\n".join(summaries))                                                                          │
│                                                                                                                 │
│ Example — recursive binary split for very large contexts:                                                       │
│     mid   = len(rlm_context) // 2                                                                               │
│     left  = rlm_call("Analyse first half",  rlm_context[:mid\])                                                 │
│     right = rlm_call("Analyse second half", rlm_context[mid:\])                                                 │
│     final_answer(left + " " + right)                  

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Split rlm_context by SECTION: headers                                                                          
  lines = rlm_context.split('\n')                                                                                  
  sections = []                                                                                                    
  current_section = []                                                                                             
  current_name = ""                                                                                                
                                                                                                                   
  for line in lines:                                                                                               
      if line.strip().startswith('SECTION:'):                                                                      
          # Save previous section if exists                                                                        
          if current_section:                                                                                      
              sections.append({                                                                                    
                  'name': current_name,                                                                            
                  'content': '\n'.join(current_section)                                                            
              })                                                                                                   
          # Start new section                                                                                      
          current_name = line.strip().replace('SECTION:', '').strip()                                              
          current_section = []                                                                                     
      else:                                                                                                        
          current_section.append(line)                                                                             
                                                                                                                   
  # Don't forget the last section                                                                                  
  if current_section:                                                                                              
      sections.append({                                                                                            
          'name': current_name,                                                                                    
          'content': '\n'.join(current_section)                                                                    
      })                                                                                                           
                                                                                                                   
  print(f"Found {len(sections)} sections")                                                                         
  for i, s in enumerate(sections):                                                                                 
      print(f"Section {i+1}: {s['name'][:50]}... (preview: {s['content'][:80]})")                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Found 7 sections
Section 1: ... (preview: 
============================================================)
Section 2: Financial Performance... (preview: ============================================================

The fiscal year co)
Section 3: Human Resources & Talent... (preview: ============================================================

The talent-acquisi)
Section 4: Research & Development... (preview: ============================================================

The research divis)
Section 5: Operations & Infrastructure... (preview: ============================================================

The infrastructure)
Section 6: Customer Experience... (preview: ============================================================

Customer-engagemen)
Section 7: Legal & Compliance... (preview: ============================================================

The legal departme)

Out: None

[Step 1: Duration 11.17 seconds| Input tokens: 3,503 | Output tokens: 363]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Section: Total quarterly revenue reached $473.8 million, reflecting an eleven percent              
  year-over-year increase.\n\nFinancial Performance: Total quarterly revenue reached $473.8 million, reflecting    
  an eleven percent year-over-year increase.\n\nHuman Resources & Talent: Representation of women in senior        
  leadership roles increased to 34 percent, demonstrating continued progress in diversity and inclusion that       
  earned recognition from several external benchmarking organisations.\n\nResearch & Development: The research     
  division demonstrated a 40% reduction in inference latency for production workloads through a successful         
  proof-of-concept on the next-generation platform architecture.\n\nOperations & Infrastructure: The               
  infrastructure team achieved a production server uptime of 99.97%, the highest figure in the company's history   
  and exceeding the 99.9% service-level agreement target.\n\nCustomer Experience: Net Promoter Score reached 72,   
  an all-time high for the company, driven by improved onboarding flow and responsive support                      
  experience.\n\nLegal & Compliance: Anti-corruption training was completed by one hundred percent of employees    
  in scope, confirming full regulatory adherence to the updated programme.")                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Section: Total quarterly revenue reached $473.8 million, reflecting an eleven percent year-over-year 
increase.

Financial Performance: Total quarterly revenue reached $473.8 million, reflecting an eleven percent year-over-year 
increase.

Human Resources & Talent: Representation of women in senior leadership roles increased to 34 percent, demonstrating
continued progress in diversity and inclusion that earned recognition from several external benchmarking 
organisations.

Research & Development: The research division demonstrated a 40% reduction in inference latency for production 
workloads through a successful proof-of-concept on the next-generation platform architecture.

Operations & Infrastructure: The infrastructure team achieved a production server uptime of 99.97%, the highest 
figure in the company's history and exceeding the 99.9% service-level agreement target.

Customer Experience: Net Promoter Score reached 72, an all-time high for the company, driven by improved onboarding
flow and responsive support experience.

Legal & Compliance: Anti-corruption training was completed by one hundred percent of employees in scope, confirming
full regulatory adherence to the updated programme.

[Step 5: Duration 7.15 seconds| Input tokens: 24,258 | Output tokens: 1,252]


EXECUTIVE SUMMARY
Section: Total quarterly revenue reached $473.8 million, reflecting an eleven percent year-over-year increase.

Financial Performance: Total quarterly revenue reached $473.8 million, reflecting an eleven percent year-over-year increase.

Human Resources & Talent: Representation of women in senior leadership roles increased to 34 percent, demonstrating continued progress in diversity and inclusion that earned recognition from several external benchmarking organisations.

Research & Development: The research division demonstrated a 40% reduction in inference latency for production workloads through a successful proof-of-concept on the next-generation platform architecture.

Operations & Infrastructure: The infrastructure team achieved a production server uptime of 99.97%, the highest figure in the company's history and exceeding the 99.9% service-level agreement target.

Customer Experience: Net Promoter Score reached 72, an all-time high for the company, driven by imp

### Inspect the summarization call tree

This should show a root node that splits the document, with child nodes
for each section summary — the classic RLM recursive decomposition pattern.

In [ ]:
print('=== Summarization Call Tree ===')
print_tree(summ_result.metadata['call_tree'])

# Count children to verify sub-agents were called
n_children = len(summ_result.metadata['call_tree'].get('children', []))
print(f'\nSub-agent calls: {n_children}')
if n_children > 0:
    print('✅ Recursive decomposition confirmed — sub-agents were used!')
else:
    print('⚠️  No sub-agents called — the agent may have tried to summarise directly.')

In [7]:
# Save the summarization trace
save_html(summ_result, log_dir / 'summarization_demo.html')
save_json(summ_result, log_dir / 'summarization_demo.json')
print(f'Summarization HTML trace saved to: {log_dir / "summarization_demo.html"}')

Summarization HTML trace saved to: /workspace/logs/summarization_demo.html


---
## Experiment B — Natural-Language Comprehension Q&A

Five questions about the document. The answers are embedded in **natural
language prose** — numbers written as words, no labels, no markers. The
questions use different phrasing than the source text.

This task **cannot be solved with regex or string search**. The agent must
have sub-agents read and comprehend each section to find the answers.

In [ ]:
questions_block = '\n'.join(
    f"{qid}. {info['question']}"
    for qid, info in GROUND_TRUTH.items()
)

agent_qa = make_agent(max_depth=2, max_steps=15, verbose=True)

qa_result = agent_qa.completion(
    task=(
        "`rlm_context` is a long corporate annual report divided into sections "
        "(each section begins with 'SECTION: <name>' between lines of '=' characters). Answer the following questions by searching "
        "the document.\n\n"
        f"Questions:\n{questions_block}\n\n"
        "IMPORTANT — Read carefully:\n"
        "- The answers are written in natural language prose within the report. "
        "Numbers may be written as words (e.g., 'four hundred seventy-three' not '473'). "
        "Dates may be written conversationally (e.g., 'September fifteenth, twenty twenty-six').\n"
        "- You CANNOT find these answers with regex or string search.\n"
        "- You MUST split the document by 'SECTION:' boundaries and use llm_call() or rlm_call() "
        "to have a sub-agent read each section and look for the answer.\n"
        "- Pass section text as context_slice, NOT embedded in the task string.\n\n"
        "Strategy:\n"
        "1. Split `rlm_context` into sections by finding lines that start with 'SECTION:'.\n"
        "2. For each section, call llm_call('Read this section and answer any of these "
        "questions that can be answered from it: [questions]', section_text).\n"
        "3. Collect all partial answers and assemble the final response.\n\n"
        "Return a numbered list matching the question IDs (Q1-Q5) with the answers "
        "converted to standard numeric format (e.g., $473.8M, 37, 99.97%, etc.)."
    ),
    context=REPORT,
)

print('\n' + '=' * 60)
print('Q&A ANSWERS')
print('=' * 60)
print(qa_result.response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are an RLM (Recursive Language Model) agent at recursion depth 0/2.                                         │
│                                                                                                                 │
│ You run inside a Python REPL.  The input context is available as the                                            │
│ Python variable `rlm_context` — treat it as a Python object.  Slice it,                                         │
│ search it, split it, transform it.  Do NOT embed its raw content as a                                           │
│ string literal inside any sub-call argument.                                                                    │
│                                                                                                                 │
│ Two tools are available for making LLM sub-calls:                                                               │
│                                                                                                                 │
│ `llm_call(sub_task, context_slice)`:                                                                            │
│     Direct, non-recursive LLM call.  Fast and lightweight.                                                      │
│     Use for leaf-level queries on chunks that are small enough to                                               │
│     answer in a single LLM call without further decomposition.                                                  │
│                                                                                                                 │
│ `rlm_call(sub_task, context_slice)`:                                                                            │
│     Recursive RLM sub-call.  The child agent gets its own Python REPL                                           │
│     and can decompose the slice further.  Use for complex sub-tasks                                             │
│     that may themselves need recursive processing.                                                              │
│                                                                                                                 │
│ You decide HOW to orchestrate — use any Python logic to split, filter,                                          │
│ or transform `rlm_context` before passing slices to sub-calls.                                                  │
│                                                                                                                 │
│ Example — summarise paragraph-by-paragraph with direct LLM calls:                                               │
│     paragraphs = [p for p in rlm_context.split("\n\n") if p.strip()\]                                           │
│     summaries = [llm_call(f"Summarise paragraph {i+1}", p)                                                      │
│                  for i, p in enumerate(paragraphs)\]                                                            │
│     final_answer("\n".join(summaries))                                                                          │
│                                                                                                                 │
│ Example — recursive binary split for very large contexts:                                                       │
│     mid   = len(rlm_context) // 2                                                                               │
│     left  = rlm_call("Analyse first half",  rlm_context[:mid\])                                                 │
│     right = rlm_call("Analyse second half", rlm_context[mid:\])                                                 │
│     final_answer(left + " " + right)                  

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # First, let's see the structure of the document                                                                 
  print("Document length:", len(rlm_context))                                                                      
  print("\nFirst 500 characters:")                                                                                 
  print(rlm_context[:500])                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Document length: 18982

First 500 characters:

============================================================
SECTION: Financial Performance
============================================================

The fiscal year concluded with a strong performance across all revenue streams. Management attributed the gains to 
disciplined execution of the multi-year growth strategy and favourable market conditions in key geographies.

Gross margins expanded by approximately one hundred and twenty basis points compared with the prior-year period, 
driven 

Out: None

[Step 1: Duration 4.39 seconds| Input tokens: 3,558 | Output tokens: 211]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Compile all answers from the sections                                                                          
  final_answers = {                                                                                                
      "Q1": "$473.8 million",  # From Financial Performance                                                        
      "Q2": "Dr. Ryland Grace",  # From Human Resources & Talent                                                   
      "Q3": "42",  # From Research & Development                                                                   
      "Q4": "99.97%",  # From Operations & Infrastructure                                                          
      "Q5": "September 15, 2026"  # From Customer Experience                                                       
  }                                                                                                                
                                                                                                                   
  # Format the final answer as a numbered list                                                                     
  final_response = []                                                                                              
  for i in range(1, 6):                                                                                            
      q_key = f"Q{i}"                                                                                              
      answer = final_answers[q_key]                                                                                
      final_response.append(f"{q_key}. {answer}")                                                                  
                                                                                                                   
  final_answer("\n".join(final_response))                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Q1. $473.8 million
Q2. Dr. Ryland Grace
Q3. 42
Q4. 99.97%
Q5. September 15, 2026

[Step 7: Duration 8.91 seconds| Input tokens: 35,479 | Output tokens: 2,009]


Q&A ANSWERS
Q1. $473.8 million
Q2. Dr. Ryland Grace
Q3. 42
Q4. 99.97%
Q5. September 15, 2026


### Inspect the Q&A call tree

In [ ]:
print('=== Q&A Call Tree ===')
print_tree(qa_result.metadata['call_tree'])

n_children = len(qa_result.metadata['call_tree'].get('children', []))
print(f'\nSub-agent calls: {n_children}')
if n_children > 0:
    print('✅ Recursive decomposition confirmed — sub-agents were used!')
else:
    print('⚠️  No sub-agents called.')

---
## Step 3 — Verify accuracy

Check whether the RLM's answers contain the ground-truth values.

In [8]:
answer = qa_result.response.lower()

print('=== Accuracy Check ===')
found = 0
for qid, info in GROUND_TRUTH.items():
    matched = any(kw.lower() in answer for kw in info['keywords'])
    if matched:
        found += 1
        print(f'  ✅ {qid}: {info["question"]}')
        print(f'        Expected: {info["answer"]}')
    else:
        print(f'  ❌ {qid}: {info["question"]}')
        print(f'        Expected: {info["answer"]}  — NOT FOUND in answer')

print(f'\nScore: {found}/5 questions answered correctly')
if found == 5:
    print('🎉 Perfect score — all answers found through recursive comprehension!')
elif found >= 3:
    print('👍 Good performance — most answers found.')
else:
    print('⚠️  The model may need more steps or stronger prompting.')

NameError: name 'qa_result' is not defined

---
## Step 4 — View agent code steps

See exactly what Python code the agents wrote at each level of the recursion.

In [ ]:
def print_agent_steps(node: dict, indent: int = 0) -> None:
    prefix = '  ' * indent
    depth = node.get('depth', '?')
    task = node.get('task_preview', node.get('task', '')[:120])
    steps = node.get('agent_steps', [])
    if steps:
        print(f'{prefix}── Node depth={depth}: {task}')
        for s in steps:
            num = s.get('step_number', '?')
            code_action = s.get('code_action', '')
            obs = s.get('observations', '')
            is_final = s.get('is_final_answer', False)
            print(f"{prefix}  Step {num} {'[FINAL]' if is_final else ''}:")
            if code_action:
                for line in code_action.split('\n'):
                    print(f'{prefix}    >>> {line}')
            if obs:
                for line in str(obs).split('\n')[:5]:
                    print(f'{prefix}    ... {line}')
        print()
    for child in node.get('children', []):
        print_agent_steps(child, indent + 1)

print('=== Q&A Agent Steps ===')
print_agent_steps(qa_result.metadata['call_tree'])

---
## Step 5 — Save interactive HTML visualizations

Open the generated files in any browser to interactively explore the
recursive call trees, agent steps, and LLM request payloads.

In [19]:
# Save Q&A trace
save_html(qa_result, log_dir / 'qa_comprehension_demo.html')
save_json(qa_result, log_dir / 'qa_comprehension_demo.json')
print(f'Q&A HTML trace saved to: {log_dir / "qa_comprehension_demo.html"}')
print(f'Q&A JSON trace saved to: {log_dir / "qa_comprehension_demo.json"}')
print()
print('Open the HTML files in a browser to explore the recursive call tree interactively.')

Q&A HTML trace saved to: /workspace/logs/qa_comprehension_demo.html
Q&A JSON trace saved to: /workspace/logs/qa_comprehension_demo.json

Open the HTML files in a browser to explore the recursive call tree interactively.


---
## What to try next

- **Increase document size**: Add more sections or more paragraphs per section.
  Larger documents make recursive decomposition even more necessary.
- **Change `max_depth`**: Try `max_depth=1` (shallower) vs `max_depth=3` (deeper)
  and see how the call tree changes.
- **Add harder questions**: Questions requiring cross-section reasoning (e.g.,
  "Which investment — R&D or infrastructure — had the larger impact?")
  would demonstrate multi-hop recursive Q&A.
- **Compare with a plain LLM**: Use `max_depth=0` and see if a single-shot
  LLM can answer the questions when the document is too large for its
  context window.
- **Try the visualizer**: Open the saved HTML files to explore the full
  recursive trace interactively.

### Key takeaway

The hierarchical summarization experiment demonstrates the core RLM paper insight:
**a model that can recursively decompose tasks, delegate sub-tasks to child agents,
and aggregate results can handle inputs of arbitrary length** — something no
single-shot LLM can do.